In [1]:

# =========================
# 0) Imports + Config
# =========================
import os, re, math, glob, random, unicodedata, zlib, time
from dataclasses import dataclass
from contextlib import nullcontext
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Sequence, Set, Tuple, Union
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import joblib

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed
from transformers import Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, Seq2SeqTrainer
from tqdm.auto import tqdm

import sacrebleu
from sacrebleu.metrics import BLEU, CHRF

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

print("torch:", torch.__version__)
import transformers
print("transformers:", transformers.__version__)
print("sacrebleu:", sacrebleu.__version__)

# Kaggle-friendly cache dirs (optional)
os.environ.setdefault("HF_HOME", "/kaggle/working/hf_home")
os.environ.setdefault("HF_DATASETS_CACHE", "/kaggle/working/hf_datasets_cache")
os.environ.setdefault("TRANSFORMERS_CACHE", "/kaggle/working/hf_transformers_cache")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# -------------------------
# Config (edit these)
# -------------------------
@dataclass
class Config:
    SEED: int = 4213

    # Competition data
    INPUT_DIR: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
    TEST_PATH: str = INPUT_DIR + '/test.csv'
    SUB_PATH: str  = INPUT_DIR + '/sample_submission.csv'

    # Model directory (attach model as Kaggle Dataset and set this if auto-discovery fails)
    OUTPUT_DIR: str = '/kaggle/working'

    MODEL_DIR: str = "/kaggle/input/models/mattiaangeli/akkadian-model-mind-the-gap/pytorch/default/8/checkpoint-1548" 

    # Prefix used during training (keep if your model was trained with it)
    PREFIX: str = "translate Akkadian to English: "

    # Tokenization / generation lengths
    SRC_MAX_LENGTH: int = 512
    GEN_MAX_NEW_TOKENS: int = 512

    # Decode pool (MBR-style)
    USE_MBR: bool = True
    MBR_POOL_CAP: int = 36

    # Beam candidates
    NUM_BEAMS: int = 4
    NUM_BEAM_CANDS: int = 1
    MBR_GLOSS_VARIANTS: int = 1
    LENGTH_PENALTY: float = 1.2

    # Sampling candidates
    NUM_SAMPLE_CANDS: int = 6
    TEMPERATURE: float = 0.75
    TOP_P: float = 0.9

    REPETITION_PENALTY: float = 1.1
    NO_REPEAT_NGRAM: int = 0

    # Inference batch sizes
    BATCH_EXAMPLES: int = 16  # outer batch over examples
    BATCH_INPUTS: int = 16    # inner batch over variant-inputs for generate()

    # Optional: TBM retrieval (adds train translations into the pool)
    TBM_ENABLE: bool = True
    TBM_CACHE_PATH: str = "/kaggle/working/tbm_index.joblib"
    TBM_TOPK: int = 3
    TBM_MIN_SIM: float = 0.90
    TBM_HARD_SIM: float = 0.995
    TBM_NGRAM_MIN: int = 3
    TBM_NGRAM_MAX: int = 6
    TBM_MAX_FEATURES: int = 250_000
    TBM_N_NEIGHBORS: int = 8

    # Optional: glossary augmentation (adds "GLOSSARY: ..." to the source)
    GLOSS_ENABLE: bool = True
    GLOSSER_JL_PATH: str = os.path.join(MODEL_DIR, "glosser.joblib")
    GLOSS_MAX_ITEMS: int = 4

    # These lexicon files are shipped with the competition (usually in INPUT_DIR)
    LEXICON_PATH: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv"
    EBL_DICT_PATH: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/eBL_Dictionary.csv"
    TBM_PAIRS_PATH: str = os.path.join(MODEL_DIR, "tbm_pairs.csv")

    # Optional: MSA consensus polishing on top of MBR (safe: includes n-gram + MBR verification)
    USE_MSA_POLISH: bool = False
    MSA_GAP_THR: float | None = None  # if set: only run MSA when MBR gap <= thr
    MSA_MIN_POOL: int = 3
    MSA_MIN_AGREEMENT: float = 0.35
    MSA_TIE_BIAS: int = 2
    MSA_FUZZY_THR: float = 0.5
    MSA_MAX_INSERT_LEN: int = 3
    MSA_NGRAM_VERIFY: bool = True
    MSA_NGRAM_MAX_N: int = 4
    MSA_NGRAM_MIN_GAIN: float = 0.0
    MSA_MBR_VERIFY: bool = True

cfg = Config()
set_seed(cfg.SEED)

# Device / dtype
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16
print("device:", device, "dtype:", dtype)

# Preprocessing utilities (SOFT-HARMONIZED — behavior switches via mode="train"/"infer")
# - Host-aligned but NOT overly strict.
# - Controversial unit/grain rewrites removed.
# - Avoid breaking hyphen-attached gaps (U-<gap>, <gap>-Aššur).
# - Unicode fractions enforced ONLY in infer mode (host: hidden test has no decimals).

# ============================================================
# Canonical decimals (float-artifact squashing) — HOST: 4 decimals
# ============================================================
_ALLOWED_FRACS = [
    (1.0 / 6.0, "0.1666"),
    (1.0 / 4.0, "0.25"),
    (1.0 / 3.0, "0.3333"),
    (1.0 / 2.0, "0.5"),
    (2.0 / 3.0, "0.6666"),
    (3.0 / 4.0, "0.75"),
    (5.0 / 6.0, "0.8333"),
]
_FRAC_TOL = 2e-3
_FLOAT_ARTIFACT_RE = re.compile(r"(?<![\w/])(\d+\.\d{6,})(?![\w/])")


def _canon_decimal_str(x: float) -> str:
    ip = int(math.floor(x + 1e-12))
    frac = x - ip
    best = None
    for v, dec in _ALLOWED_FRACS:
        d = abs(frac - v)
        if best is None or d < best[0]:
            best = (d, dec)
    if best and best[0] <= _FRAC_TOL:
        dec = best[1]
        if ip == 0:
            return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}{dec}"
    return f"{x:.4f}".rstrip("0").rstrip(".")


def normalize_float_artifacts(text: str) -> str:
    s = "" if text is None else str(text)

    def repl(m):
        raw = m.group(1)
        try:
            return _canon_decimal_str(float(raw))
        except Exception:
            return raw

    return _FLOAT_ARTIFACT_RE.sub(repl, s)


# ============================================================
# Decimals -> unicode fractions (infer-only)
# ============================================================
_DEC2UNICODE_FRAC = {
    "0.5": "½",
    "0.25": "¼",
    "0.3333": "⅓",
    "0.8333": "⅚",
    "0.625": "⅝",
    "0.6666": "⅔",
    "0.75": "¾",
    "0.1666": "⅙",
}

_DEC_TOKEN_RE = re.compile(
    r"(?<![\w/])"
    r"(\d+)\.(1666|25|3333|5|625|6666|75|8333)"
    r"(?![\w/])"
)


def decimals_to_unicode_fractions(s: str) -> str:
    if s is None:
        return ""
    t = str(s)

    def repl(m):
        ip = int(m.group(1))
        dec = f"0.{m.group(2)}"
        frac = _DEC2UNICODE_FRAC.get(dec)
        if not frac:
            return m.group(0)
        if ip == 0:
            return frac
        return f"{ip} {frac}"

    return _DEC_TOKEN_RE.sub(repl, t)


# ============================================================
# Gaps — canonicalize to "<gap>"
# ============================================================
_TAG_GAP_RE = re.compile(r"<\s*gap\s*>", re.I)
_TAG_BIGGAP_RE = re.compile(r"<\s*big[\s_\-]*gap\s*>", re.I)
_BARE_BIGGAP_RE = re.compile(r"\bbig[\s_\-]*gap\b", re.I)

_ELLIPSIS_RE = re.compile(r"(?:\.{3,}|…+|\[\.+\])")
_BRACKET_X_RE = re.compile(r"(\[\s*x\s*\]|\(\s*x\s*\))", re.I)
_XTOKEN_RUN_RE = re.compile(r"\bx(?:\s+x)+\b", re.I)
_XRUN_RE = re.compile(r"(?<!\w)x{2,}(?!\w)", re.I)
_XTOK_RE = re.compile(r"(?<!\w)x(?!\w)", re.I)

_BRACKET_LACUNA_RE = re.compile(r"\[\s*(?:x|\.|\s)+\s*\]", re.I)
_STAR_X_RE = re.compile(r"\*\s*x\b", re.I)
_BREAK_RE = re.compile(
    r"\(\s*(?:break|large\s+break|n\s+broken\s+lines|\d+\s+broken\s+lines|broken\s+lines?)\s*\)",
    re.I,
)

_WS_RE = re.compile(r"\s+")


def normalize_gaps(text: str) -> str:
    if text is None:
        return ""
    t = str(text)

    t = _TAG_BIGGAP_RE.sub("<gap>", t)
    t = _TAG_GAP_RE.sub("<gap>", t)
    t = _BARE_BIGGAP_RE.sub("<gap>", t)

    t = _BREAK_RE.sub("<gap>", t)

    t = _BRACKET_LACUNA_RE.sub("<gap>", t)
    t = _STAR_X_RE.sub("<gap>", t)

    t = _XTOKEN_RUN_RE.sub("<gap>", t)
    t = _ELLIPSIS_RE.sub("<gap>", t)
    t = _BRACKET_X_RE.sub("<gap>", t)
    t = _XRUN_RE.sub("<gap>", t)
    t = _XTOK_RE.sub("<gap>", t)

    return t


_GAP_TOKEN_RE = re.compile(r"^-?<gap>-?$", re.I)


def collapse_gap_runs_tokens(tokens: List[str], mode: str) -> List[str]:
    mode = (mode or "none").lower().strip()
    if mode in ("none", ""):
        return tokens
    if mode in ("big_only", "any2big"):
        mode = "single"

    def is_gap_tok(tok: str) -> bool:
        return bool(_GAP_TOKEN_RE.match(str(tok)))

    out = []
    i = 0
    n = len(tokens)
    while i < n:
        if is_gap_tok(tokens[i]):
            j = i
            while j < n and is_gap_tok(tokens[j]):
                j += 1
            out.append("<gap>")
            i = j
        else:
            out.append(tokens[i])
            i += 1
    return out


def space_gap_token_hyphen_safe(s: str) -> str:
    if s is None:
        return ""
    t = str(s)
    t = re.sub(r"(?<![\s\-])<gap>", " <gap>", t)
    t = re.sub(r"<gap>(?![\s\-])", "<gap> ", t)
    return t


_GAP_RUN_STR_RE = re.compile(
    r"(?i)"
    r"(?:"
    r"(?:^|(?<=\s))"
    r"-?\s*<gap>\s*-?"
    r"(?:\s*[\.,;:]\s*)?"
    r"(?:\s+|\s*$)"
    r"){2,}"
)


def collapse_gap_runs_string(s: str) -> str:
    if s is None:
        return ""
    t = str(s)

    t = re.sub(r"(?i)<gap>\s*-\s*<gap>", "<gap>-<gap>", t)
    t = re.sub(r"(?i)<gap>\s*[\.,;:]\s*<gap>", "<gap> <gap>", t)

    t = _GAP_RUN_STR_RE.sub(" <gap> ", t)
    return _WS_RE.sub(" ", t).strip()


# ============================================================
# ASCII/Oracc/ATF -> host diacritics
# ============================================================
_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a": "á", "e": "é", "i": "í", "u": "ú", "A": "Á", "E": "É", "I": "Í", "U": "Ú"})
_GRAVE = str.maketrans({"a": "à", "e": "è", "i": "ì", "u": "ù", "A": "À", "E": "È", "I": "Ì", "U": "Ù"})


def ascii_to_diacritics(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s


# ============================================================
# Determinatives alignment (host)
# ============================================================
_DET_DKI_RE = re.compile(r"\(\s*(d|ki)\s*\)", re.I)
_TUG_PARENS_RE = re.compile(r"\(\s*TÚG\s*\)")


def normalize_determinatives(s: str) -> str:
    if s is None:
        return ""
    t = str(s)
    t = _DET_DKI_RE.sub(lambda m: "{%s}" % m.group(1).lower(), t)
    t = _TUG_PARENS_RE.sub("TÚG", t)
    return t


# ============================================================
# Transliteration char cleanup (host-aligned)
# ============================================================
TRANSLIT_SPECIAL_CHAR_MAP = {
    "ḫ": "h",
    "Ḫ": "H",
    "ʾ": "",
    "₀": "0",
    "₁": "1",
    "₂": "2",
    "₃": "3",
    "₄": "4",
    "₅": "5",
    "₆": "6",
    "₇": "7",
    "₈": "8",
    "₉": "9",
    "—": "-",
    "–": "-",
}
TRANSLIT_SPECIAL_SEQ_MAP = {"mₓ": "m", "zₓ": "z"}
_SUB_X = "ₓ"
_CHAR_TRANS = str.maketrans(TRANSLIT_SPECIAL_CHAR_MAP)


def normalize_silver_abbrev(s: str) -> str:
    if s is None:
        return ""
    t = str(s)
    t = re.sub(r"\bKÙ\.B\.(?=\s|$)", "KÙ.BABBAR", t)
    t = re.sub(r"\bKÙ\.B\b", "KÙ.BABBAR", t)
    return t


def normalize_external_transliteration(text: str, *, kb_to_silver: bool = True) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    s = str(text)

    s = ascii_to_diacritics(s)
    s = normalize_determinatives(s)
    s = normalize_gaps(s)

    for k, v in TRANSLIT_SPECIAL_SEQ_MAP.items():
        s = s.replace(k, v)

    s = s.translate(_CHAR_TRANS).replace(_SUB_X, "")
    s = normalize_float_artifacts(s)

    s = normalize_silver_abbrev(s)
    if kb_to_silver:
        s = re.sub(r"\bKB\b", "KÙ.BABBAR", s)

    s = space_gap_token_hyphen_safe(s)
    s = " ".join(collapse_gap_runs_tokens(s.split(), "single"))
    s = collapse_gap_runs_string(s)

    s = _WS_RE.sub(" ", s).strip()
    return s


# ============================================================
# Translation normalizer (soft)
# ============================================================
_PN_RE = re.compile(r"\bPN\b")

_QUOTE_NORM_TRANS = str.maketrans({
    "“": '"',
    "”": '"',
    "„": '"',
    "«": '"',
    "»": '"',
    "‘": "'",
    "’": "'",
})

_SOFT_GRAM_PARENS_RE = re.compile(
    r"""
    \(
      \s*
      (?:
        fem(?:\.)? |
        sing(?:\.)? |
        plur(?:\.)? |
        pl(?:\.)? |
        singular |
        plural |
        \? |
        \!
      )
      (?:\s*(?:[.;,]?\s*(?:fem|sing|plur|pl|singular|plural)(?:\.)?)\s*)*
      \s*
    \)
    """,
    re.I | re.VERBOSE,
)

_RE_GOLD = re.compile(r"(?<!\w)-gold\b")
_RE_TAX = re.compile(r"(?<!\w)-tax\b")
_RE_TEXTILES_DASH = re.compile(r"(?i)(?<!\w)-textiles\b")


def apply_host_translation_rewrites(text: str, *, enable: bool = True) -> str:
    if not enable:
        return "" if text is None else str(text)
    s = "" if text is None else str(text)
    s = _RE_GOLD.sub("pašallum gold", s)
    s = _RE_TAX.sub("šadduātum tax", s)
    s = _RE_TEXTILES_DASH.sub("kutānum textiles", s)
    return s


def normalize_external_translation(
    text: str,
    *,
    gap_collapse: str = "single",
    enable_host_rewrites: bool = True,
) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    s = str(text)

    s = normalize_gaps(s)
    s = _PN_RE.sub("<gap>", s)

    s = _SOFT_GRAM_PARENS_RE.sub(" ", s)
    s = s.translate(_QUOTE_NORM_TRANS)

    s = normalize_float_artifacts(s)

    if enable_host_rewrites:
        s = apply_host_translation_rewrites(s, enable=True)

    if gap_collapse and gap_collapse.lower().strip() not in ("none", ""):
        toks = collapse_gap_runs_tokens(s.split(), gap_collapse)
        s = " ".join(toks)

    s = collapse_gap_runs_string(s)
    s = _WS_RE.sub(" ", s).strip()
    return s


# ============================================================
# Main transliteration preprocessor (mode switch)
# ============================================================
class OptimizedPreprocessor:
    def __init__(self, mode: str = "train"):
        self.mode = (mode or "train").lower().strip()
        self._char_trans = _CHAR_TRANS

    def preprocess_input_text(self, text: str) -> str:
        if text is None or (isinstance(text, float) and pd.isna(text)):
            return ""
        s = str(text)

        s = ascii_to_diacritics(s)
        s = normalize_determinatives(s)
        s = normalize_gaps(s)

        for k, v in TRANSLIT_SPECIAL_SEQ_MAP.items():
            s = s.replace(k, v)

        s = s.translate(self._char_trans).replace(_SUB_X, "")
        s = normalize_float_artifacts(s)

        s = normalize_silver_abbrev(s)
        s = re.sub(r"\bKB\b", "KÙ.BABBAR", s)

        s = space_gap_token_hyphen_safe(s)
        s = " ".join(collapse_gap_runs_tokens(s.split(), "single"))
        s = collapse_gap_runs_string(s)

        s = _WS_RE.sub(" ", s).strip()
        return s

    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)

        ser = ser.apply(ascii_to_diacritics)
        ser = ser.apply(normalize_determinatives)
        ser = ser.apply(normalize_gaps)

        for k, v in TRANSLIT_SPECIAL_SEQ_MAP.items():
            ser = ser.str.replace(k, v, regex=False)

        ser = ser.str.translate(self._char_trans)
        ser = ser.str.replace(_SUB_X, "", regex=False)

        ser = ser.str.replace(
            _FLOAT_ARTIFACT_RE,
            lambda m: _canon_decimal_str(float(m.group(1))),
            regex=True,
        )

        ser = ser.apply(normalize_silver_abbrev)
        ser = ser.str.replace(r"\bKB\b", "KÙ.BABBAR", regex=True)

        ser = ser.apply(space_gap_token_hyphen_safe)
        ser = ser.apply(lambda x: " ".join(collapse_gap_runs_tokens(str(x).split(), "single")))
        ser = ser.apply(collapse_gap_runs_string)

        ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()
        return ser.tolist()


# ============================================================
# Vectorized postprocessor for translations (mode switch)
# ============================================================
class VectorizedPostprocessor:
    def __init__(
        self,
        mode: str = "infer",
        *,
        aggressive: bool = True,
        empty_fallback: str = "",
        fix_repeats: bool = False,
        enable_gap_run_collapse: bool = True,
        enable_host_rewrites: bool = True,
    ):
        self.mode = (mode or "infer").lower().strip()
        self.aggressive = bool(aggressive)
        self.fix_repeats = bool(fix_repeats)
        self.empty_fallback = "" if empty_fallback is None else str(empty_fallback)

        self.enable_unicode_fractions = (self.mode == "infer")
        self.enable_gap_run_collapse = bool(enable_gap_run_collapse)
        self.enable_host_rewrites = bool(enable_host_rewrites)

        self.gap_collapse = "single" if self.mode == "infer" else "single"
        self._pn_re = _PN_RE
        self._soft_gram_parens_re = _SOFT_GRAM_PARENS_RE
        self._quote_norm_trans = _QUOTE_NORM_TRANS

        self.forbidden_chars = "—–<>⌈⌋⌊+ʾ"
        self.forbidden_trans = str.maketrans("", "", self.forbidden_chars)

        self.patterns = {
            "gap_legacy": re.compile(r"(\[x\]|\(x\)|\bx\b)", re.I),
            "big_gap_legacy": re.compile(r"(\.{3,}|…|\[\.+\])"),
            "whitespace": _WS_RE,
            "punct_space": re.compile(r"\s+([.,:;])"),
            "repeated_punct": re.compile(r"([.,:;])\1+"),
            "repeated_words": re.compile(r"\b(\w+)(?:\s+\1\b)+"),
        }

        self._month_roman_re = re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.IGNORECASE)
        self._roman2int = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}

    def _month_repl(self, m):
        r = m.group(1).upper()
        return f"Month {self._roman2int.get(r, r)}"

    def _collapse_gaps_str(self, t: str) -> str:
        toks = collapse_gap_runs_tokens(str(t).split(), self.gap_collapse)
        return " ".join(toks)

    def postprocess_batch(self, translations: List[str]) -> List[str]:
        s = pd.Series(translations)

        valid_mask = s.apply(lambda x: isinstance(x, str) and (len(x.strip()) > 0))
        if not bool(valid_mask.all()):
            s.loc[~valid_mask] = self.empty_fallback
            
        s = s.str.replace(self._month_roman_re, self._month_repl, regex=True)
        s = s.apply(normalize_gaps)
        s = s.str.replace(self._pn_re, "<gap>", regex=True)
        s = s.apply(lambda x: ("" if x is None else str(x)).translate(self._quote_norm_trans))
        s = s.str.replace(self.patterns["whitespace"], " ", regex=True).str.strip()

        if self.aggressive:
            s = s.str.replace(self.patterns["gap_legacy"], "<gap>", regex=True)
            s = s.str.replace(self.patterns["big_gap_legacy"], "<gap>", regex=True)

            s = s.str.replace(self._soft_gram_parens_re, " ", regex=True)

            s = s.apply(self._collapse_gaps_str)

            s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
            s = s.str.translate(self.forbidden_trans)
            s = s.str.replace("\x00GAP\x00", "<gap>", regex=False)

            s = s.str.replace(
                _FLOAT_ARTIFACT_RE,
                lambda m: _canon_decimal_str(float(m.group(1))),
                regex=True,
            )

            if self.enable_unicode_fractions:
                s = s.apply(decimals_to_unicode_fractions)

            if self.enable_host_rewrites:
                s = s.apply(lambda x: apply_host_translation_rewrites(x, enable=True))

            

            if self.enable_gap_run_collapse:
                s = s.apply(collapse_gap_runs_string)

            s = s.apply(space_gap_token_hyphen_safe)

            if self.fix_repeats:
                s = s.str.replace(self.patterns["repeated_words"], r"\1", regex=True)

            s = s.str.replace(self.patterns["punct_space"], r"\1", regex=True)
            s = s.str.replace(self.patterns["repeated_punct"], r"\1", regex=True)
            s = s.str.replace(self.patterns["whitespace"], " ", regex=True).str.strip()

        if self.empty_fallback:
            s = s.replace("", self.empty_fallback)

        return s.tolist()

# =========================
# 1) TBM + MBR + Trainer (inference; mimics training)
# =========================
class TBMIndex:
    """Char n-gram TF-IDF over normalized SOURCE; returns (train_translation, cosine_sim)."""
    def __init__(
        self,
        train_src_norm: list[str],
        train_tgt: list[str],
        *,
        ngram=(3, 6),
        max_features=250_000,
        n_neighbors: int = 8,
    ):
        self.train_tgt = list(map(str, train_tgt))
        self.n_neighbors = int(n_neighbors)

        self.vec = TfidfVectorizer(
            analyzer="char",
            ngram_range=tuple(ngram),
            min_df=1,
            max_features=int(max_features),
        )
        X = self.vec.fit_transform(list(map(str, train_src_norm)))
        self.nn = NearestNeighbors(n_neighbors=self.n_neighbors, metric="cosine", algorithm="brute")
        self.nn.fit(X)

    def query(self, src_norm: str, k: int = 3):
        k = max(1, min(int(k), self.n_neighbors))
        q = self.vec.transform([str(src_norm)])
        dists, idxs = self.nn.kneighbors(q, n_neighbors=k, return_distance=True)
        sims = 1.0 - dists[0]
        idxs = idxs[0]
        return [(self.train_tgt[i], float(s)) for i, s in zip(idxs, sims)]

def build_tbm_from_pairs(
    pre: "OptimizedPreprocessor",
    tbm_pairs: pd.DataFrame,
    *,
    ngram=(3, 6),
    max_features=250_000,
    n_neighbors: int = 8,
    min_pairs: int = 10,
) -> TBMIndex | None:
    """
    Build TBMIndex from a pairs DataFrame with columns:
      - transliteration
      - translation
    This matches the training notebook behavior (train-split pairs, leak-safe).
    """
    if tbm_pairs is None or len(tbm_pairs) == 0:
        return None
    if not {"transliteration","translation"}.issubset(set(tbm_pairs.columns)):
        return None

    src_raw = tbm_pairs["transliteration"].astype(str).tolist()
    tgt_raw = tbm_pairs["translation"].astype(str).tolist()

    src_norm = pre.preprocess_batch(src_raw)

    # drop empties + dedupe by normalized source (keep first)
    seen = set()
    src2, tgt2 = [], []
    for s, t in zip(src_norm, tgt_raw):
        s = _norm_ws(s)
        t = _norm_ws(t)
        if not s or not t:
            continue
        if s in seen:
            continue
        seen.add(s)
        src2.append(s)
        tgt2.append(t)

    if len(src2) < int(min_pairs):
        return None

    return TBMIndex(
        src2,
        tgt2,
        ngram=tuple(ngram),
        max_features=int(max_features),
        n_neighbors=int(n_neighbors),
    )

# ---- MBR utilities (same as training pick)
_BLEU_SENT = BLEU()
_CHRFPP_SENT = CHRF(word_order=2)

def _norm_ws(s: str) -> str:
    return " ".join(str(s).strip().split())

def _stable_int_id(s: str) -> int:
    return int(zlib.adler32(str(s).encode("utf-8")) & 0x7FFFFFFF)

def _dedup_keep_order(xs):
    seen = set()
    out = []
    for x in xs:
        x = str(x)
        if x and x not in seen:
            out.append(x)
            seen.add(x)
    return out

def _geo_sim_sentence(a: str, b: str) -> float:
    # treat b as reference for a (symmetric enough for MBR purposes when averaged)
    bleu = _BLEU_SENT.sentence_score(str(a), [str(b)]).score
    chrf = _CHRFPP_SENT.sentence_score(str(a), [str(b)]).score
    return float(math.sqrt(max(0.0, float(bleu)) * max(0.0, float(chrf))))

#  Trainer
class MBRGlossSeq2SeqTrainer(Seq2SeqTrainer):
    def __init__(
        self,
        *args,
        val_text_ds=None,
        pre=None,
        prefix="",
        post=None,            # preds post (should match inference)
        post_ref=None,        # DEBUG only (NOT used for scoring)

        glosser=None,
        gloss_variants=1,
        gloss_seed=12345,
        gloss_max_items=6,
        gloss_max_append_chars=240,

        mbr_batch_size_inputs=16,
        src_max_length=512,
        max_new_tokens=512,
        num_beams=8,
        num_beam_cands=1,
        num_sample_cands=4,
        length_penalty=1.3,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.0,
        no_repeat_ngram_size=0,
        mbr_pool_cap=10,
        show_progress=True,

        # ---- PN view (optional) ----
        canon=None,                  # SourceCanonicalizer-like, must implement canonicalize_source(text, mode=...)
        pn_enable=False,
        pn_mode="pn_norm",

        # ---- TBM (optional) ----
        tbm_index=None,              # TBMIndex, if already built
        tbm_pairs=None,              # pd.DataFrame of train-split pairs to build TBM (leak-safe)
        tbm_ngram=(3, 6),
        tbm_max_features=250_000,
        tbm_topk=3,
        tbm_min_sim=0.92,
        tbm_hard_sim=0.97,
        tbm_enable=True,

        # ---- kNN (optional; sklearn NN over encoder meanpool) ----
        knn_mem=None,                # dict from build_or_load_knn_memory_sklearn(...), optional
        knn_nn=None,                 # sklearn NearestNeighbors, optional
        knn_tgts=None,               # list[str] same order as knn_nn bank, optional
        knn_enable: bool = False,
        knn_topk: int = 8,
        knn_hint_k: int = 2,
        knn_hint_max_chars: int = 240,
        knn_ret_k: int = 1,          # inject top-R retrieved targets directly into pool
        knn_prefix_for_encode: str | None = None,  # must match how bank was embedded
        knn_query_bs: int | None = None,
        knn_min_sim: float = 0.90,
        knn_hard_sim: float = 0.94,

        # ---- MSA polish (optional) ----
        msa_enable: bool = False,          # default disabled
        msa_gap_thr: float | None = None,  # if set, only run MSA when MBR gap <= thr
        msa_min_pool: int = 3,             # minimum pool size to attempt MSA

        # ---- tag-aware MBR biases (optional) ----
        mbr_tag_prior=0.35,
        mbr_beam_penalty=0.00,
        mbr_samp_bonus=0.00,
        mbr_gloss_penalty=0.2,
        mbr_pn_penalty=0.25,
        mbr_knn_penalty=0.2,
        mbr_tbm_bonus=0.5,
        mbr_raw_bonus=0.00,

        **kwargs
    ):
        super().__init__(*args, **kwargs)
        if val_text_ds is None:
            raise ValueError("MBRGlossSeq2SeqTrainer requires val_text_ds.")
        if pre is None:
            raise ValueError("MBRGlossSeq2SeqTrainer requires pre.")

        self.val_text_ds = val_text_ds
        self.pre = pre
        self.prefix = str(prefix)

        self.post = post
        self.post_ref = post_ref  # debug only

        # PN
        self.canon = canon
        self.pn_enable = bool(pn_enable)
        self.pn_mode = str(pn_mode)

        # Gloss
        self.glosser = glosser
        self.gloss_variants = int(gloss_variants)
        self.gloss_seed = int(gloss_seed)
        self.gloss_max_items = int(gloss_max_items)
        self.gloss_max_append_chars = int(gloss_max_append_chars)

        # Decode
        self.mbr_batch_size_inputs = int(mbr_batch_size_inputs)
        self.src_max_length = int(src_max_length)
        self.max_new_tokens = int(max_new_tokens)

        self.num_beams = int(num_beams)
        self.num_beam_cands = int(num_beam_cands)
        self.num_sample_cands = int(num_sample_cands)
        self.length_penalty = float(length_penalty)
        self.temperature = float(temperature)
        self.top_p = float(top_p)
        self.repetition_penalty = float(repetition_penalty)
        self.no_repeat_ngram_size = int(no_repeat_ngram_size)

        self.mbr_pool_cap = None if mbr_pool_cap is None else int(mbr_pool_cap)
        self.show_progress = bool(show_progress)

        # ---- TBM config ----
        self.tbm_enable = bool(tbm_enable)
        self.tbm_topk = int(tbm_topk)
        self.tbm_min_sim = float(tbm_min_sim)
        self.tbm_hard_sim = float(tbm_hard_sim)

        self.tbm_index = None
        if self.tbm_enable:
            if tbm_index is not None:
                self.tbm_index = tbm_index
            elif tbm_pairs is not None:
                try:
                    self.tbm_index = build_tbm_from_pairs(
                        self.pre, tbm_pairs,
                        ngram=tuple(tbm_ngram),
                        max_features=int(tbm_max_features),
                    )
                except Exception as e:
                    self.tbm_index = None
                    if self.show_progress:
                        print("[TBM] build failed -> TBM disabled:", repr(e), flush=True)

        # ---- kNN config ----
        self.knn_enable = bool(knn_enable)
        self.knn_topk = int(knn_topk)
        self.knn_hint_k = int(knn_hint_k)
        self.knn_hint_max_chars = int(knn_hint_max_chars)
        self.knn_ret_k = int(knn_ret_k)
        self.knn_min_sim = float(knn_min_sim)
        self.knn_hard_sim = float(knn_hard_sim)

        if knn_prefix_for_encode is None:
            knn_prefix_for_encode = self.prefix
        self.knn_prefix_for_encode = str(knn_prefix_for_encode)

        if knn_query_bs is None:
            knn_query_bs = 128  # train default; override in Config if needed
        self.knn_query_bs = int(knn_query_bs)

        self.knn_nn = None
        self.knn_tgts = None
        if self.knn_enable:
            if knn_mem is not None:
                self.knn_nn = knn_mem.get("nn", None)
                self.knn_tgts = knn_mem.get("tgts", None)
            if self.knn_nn is None and knn_nn is not None:
                self.knn_nn = knn_nn
            if self.knn_tgts is None and knn_tgts is not None:
                self.knn_tgts = knn_tgts

            if self.knn_nn is None or self.knn_tgts is None:
                self.knn_enable = False
                if self.show_progress:
                    print("[kNN] disabled: missing knn_nn/knn_tgts (or knn_mem).", flush=True)

        # ---- MSA config ----
        self.msa_enable = bool(msa_enable)
        self.msa_gap_thr = None if msa_gap_thr is None else float(msa_gap_thr)
        self.msa_min_pool = int(msa_min_pool)

        # ---- tag-aware MBR biases ----
        self.mbr_tag_prior = float(mbr_tag_prior)
        self.mbr_beam_penalty = float(mbr_beam_penalty)
        self.mbr_samp_bonus = float(mbr_samp_bonus)
        self.mbr_gloss_penalty = float(mbr_gloss_penalty)
        self.mbr_pn_penalty = float(mbr_pn_penalty)
        self.mbr_knn_penalty = float(mbr_knn_penalty)
        self.mbr_tbm_bonus = float(mbr_tbm_bonus)
        self.mbr_raw_bonus = float(mbr_raw_bonus)

        # prefer caching in generate
        try:
            self.model.config.use_cache = True
        except Exception:
            pass
        try:
            if getattr(self.model, "generation_config", None) is not None:
                self.model.generation_config.use_cache = True
        except Exception:
            pass

        self.has_gloss = (self.glosser is not None) and (self.gloss_variants > 0)
        self.has_pn = self.pn_enable and (self.canon is not None)
        self.has_tbm = bool(tbm_enable) and (self.tbm_index is not None)
        self.has_knn = bool(self.knn_enable and (self.knn_nn is not None) and (self.knn_tgts is not None))

    # -------------------------
    # small helpers
    # -------------------------
    def _is_dist(self):
        return torch.distributed.is_available() and torch.distributed.is_initialized()

    def _broadcast_metrics(self, metrics: dict):
        if not self._is_dist():
            return metrics
        obj = [metrics if self.is_world_process_zero() else None]
        torch.distributed.broadcast_object_list(obj, src=0)
        return obj[0]

    def _bf16_eval_context(self):
        if torch.cuda.is_available() and bool(getattr(self.args, "bf16", False)):
            return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
        return nullcontext()

    def _tqdm(self, total: int, desc: str):
        if (not self.show_progress) or (tqdm is None):
            return None
        return tqdm(total=total, desc=desc, leave=False)

    # -------------------------
    # LEAN generator helpers (NO globals)
    # -------------------------
    def _encode_for_generate(self, tok, batch_in, *, device, src_max_length: int):
        enc = tok(
            batch_in,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=int(src_max_length),
        )
        return {k: v.to(device, non_blocking=True) for k, v in enc.items()}

    @torch.inference_mode()
    def _generate_multi_decode(self, model, tok, batch_in, *, device):
        """
        Uses self.* decode params. Returns list[list[str]]: beams first then samples.
        """
        enc = self._encode_for_generate(tok, batch_in, device=device, src_max_length=self.src_max_length)
        B = len(batch_in)
        outs = [[] for _ in range(B)]

        Rb = int(max(0, self.num_beam_cands))
        if Rb > 0:
            nb = int(max(1, int(self.num_beams), Rb))
            seq = model.generate(
                **enc,
                do_sample=False,
                num_beams=nb,
                num_return_sequences=Rb,
                max_new_tokens=int(self.max_new_tokens),
                length_penalty=float(self.length_penalty),
                repetition_penalty=float(self.repetition_penalty),
                no_repeat_ngram_size=int(self.no_repeat_ngram_size),
                use_cache=True,
                early_stopping=True,
            )
            txt = tok.batch_decode(seq, skip_special_tokens=True)
            for i in range(B):
                outs[i].extend(txt[i * Rb : (i + 1) * Rb])
            del seq, txt

        Rs = int(max(0, self.num_sample_cands))
        if Rs > 0:
            seq = model.generate(
                **enc,
                do_sample=True,
                num_beams=1,
                num_return_sequences=Rs,
                max_new_tokens=int(self.max_new_tokens),
                temperature=float(self.temperature),
                top_p=float(self.top_p),
                repetition_penalty=float(self.repetition_penalty),
                no_repeat_ngram_size=int(self.no_repeat_ngram_size),
                use_cache=True,
            )
            txt = tok.batch_decode(seq, skip_special_tokens=True)
            for i in range(B):
                outs[i].extend(txt[i * Rs : (i + 1) * Rs])
            del seq, txt

        return outs

    @torch.inference_mode()
    def _generate_multi_decode_tagged(self, model, tok, batch_in, batch_view, *, device):
        """
        Returns (cand_txt_lists, cand_tag_lists) with tags beam|<view>, samp|<view>.
        """
        views = list(batch_view)
        cands = self._generate_multi_decode(model, tok, batch_in, device=device)
        Rb = int(max(0, self.num_beam_cands))

        out_txt, out_tag = [], []
        for vw, xs in zip(views, cands):
            xs = list(xs or [])
            tags = [("beam" if k < Rb else "samp") + "|" + str(vw) for k in range(len(xs))]
            out_txt.append(xs)
            out_tag.append(tags)
        return out_txt, out_tag

    # ---- tag parsing (local) ----
    def _origin_from_tag_local(self, tag: str) -> str:
        t = "" if tag is None else str(tag)
        o = t.split("|", 1)[0] if "|" in t else t
        if o in ("beam", "samp", "tbm", "knn"):
            return o
        return o or "other"

    def _view_from_tag_local(self, tag: str) -> str:
        t = "" if tag is None else str(tag)
        if "|" in t:
            o, v = t.split("|", 1)
        else:
            o, v = t, ""
        if o == "tbm":
            return "raw"
        if o == "knn" and (not v):
            return "raw"  # retrieval inject
        v = v or "raw"
        if v.startswith("gloss"):
            return "gloss"
        if v.startswith("knn"):
            return "knn"
        if v == "pn":
            return "pn"
        if v == "raw":
            return "raw"
        return v or "other"

    def _tags_for_pick(self, ts: list[str]) -> list[str]:
        out = []
        for t in ts:
            t = "" if t is None else str(t)
            if t.startswith("tbm|"):
                out.append("tbm|raw")
            elif t.startswith("knn|"):
                out.append("knn|raw")
            else:
                o = self._origin_from_tag_local(t)
                v = self._view_from_tag_local(t)
                out.append(f"{o}|{v}")
        return out

    def _pick_mbr(self, xs: list[str], ts_pick: list[str]):
        # keep global tagged picker optional, but no globals lookup required for generation anymore
        fn = globals().get("_mbr_pick_geo_tagged", None)
        if callable(fn):
            try:
                return fn(
                    xs, ts_pick,
                    tag_prior=self.mbr_tag_prior,
                    beam_penalty=self.mbr_beam_penalty,
                    samp_bonus=self.mbr_samp_bonus,
                    gloss_penalty=self.mbr_gloss_penalty,
                    pn_penalty=self.mbr_pn_penalty,
                    knn_penalty=self.mbr_knn_penalty,
                    tbm_bonus=self.mbr_tbm_bonus,
                    raw_bonus=self.mbr_raw_bonus,
                )
            except TypeError:
                return fn(
                    xs, ts_pick,
                    tag_prior=self.mbr_tag_prior,
                    beam_penalty=self.mbr_beam_penalty,
                    samp_bonus=self.mbr_samp_bonus,
                    gloss_penalty=self.mbr_gloss_penalty,
                    pn_penalty=self.mbr_pn_penalty,
                    tbm_bonus=self.mbr_tbm_bonus,
                    raw_bonus=self.mbr_raw_bonus,
                )

        n = len(xs)
        if n <= 1:
            return 0, 0.0, 0.0
        sums = np.zeros((n,), dtype=np.float32)
        for i in range(n):
            ai = xs[i]
            for j in range(i + 1, n):
                s = _geo_sim_sentence(ai, xs[j])
                sums[i] += s
                sums[j] += s
        avg = sums / float(n - 1)
        jbest = int(np.argmax(avg))
        best = float(avg[jbest])
        tmp = avg.copy()
        tmp[jbest] = -1e9
        gap = float(best - float(np.max(tmp)))
        return jbest, gap, gap

    # -------------------------
    # kNN helpers (unchanged)
    # -------------------------
    def _knn_compute(self, src_clean: list[str]):
        if (not self.has_knn) or (self.knn_nn is None) or (self.knn_tgts is None):
            return None, None, None
        tok = getattr(self, "processing_class", None) or getattr(self, "tokenizer", None)
        if tok is None:
            return None, None, None
        dev = self.model.device
        device_str = dev.type if hasattr(dev, "type") else str(dev)
        with torch.inference_mode(), self._bf16_eval_context():
            q_emb = encode_src_meanpool(
                self.model, tok, src_clean,
                prefix=self.knn_prefix_for_encode,
                device=device_str,
                batch_size=int(self.knn_query_bs),
                max_length=int(self.src_max_length),
                use_bf16=bool(getattr(self.args, "bf16", False)),
            )
        knn_ids, knn_sims = knn_search_sklearn(self.knn_nn, q_emb, k=int(self.knn_topk))

        hints = [""] * len(src_clean)
        cap = int(self.knn_hint_max_chars)
        min_sim = float(self.knn_min_sim)
        hard_sim = float(self.knn_hard_sim)
        K = int(self.knn_hint_k)

        for i in range(len(src_clean)):
            if float(knn_sims[i, 0]) < min_sim:
                continue
            if float(knn_sims[i, 0]) >= hard_sim:
                j0 = int(knn_ids[i, 0])
                if 0 <= j0 < len(self.knn_tgts):
                    tt = str(self.knn_tgts[j0]).strip()
                    if tt:
                        hints[i] = tt[:cap]
                continue
            ts = []
            for j, sim in zip(knn_ids[i, :K], knn_sims[i, :K]):
                if float(sim) < min_sim:
                    continue
                jj = int(j)
                if 0 <= jj < len(self.knn_tgts):
                    tt = str(self.knn_tgts[jj]).strip()
                    if tt and tt not in ts:
                        ts.append(tt)
            if ts:
                hints[i] = (" || ".join(ts))[:cap]

        return knn_ids, knn_sims, hints

    def _knn_inject_retrieval(self, pools_txt, pools_tag, knn_ids, knn_sims):
        if (not self.has_knn) or knn_ids is None or knn_sims is None:
            return 0
        if int(self.knn_ret_k) <= 0:
            return 0
        hit = 0
        R = int(self.knn_ret_k)
        min_sim = float(self.knn_min_sim)
        for i in range(len(pools_txt)):
            if float(knn_sims[i, 0]) < min_sim:
                continue
            added = 0
            for j, sim in zip(knn_ids[i, :R], knn_sims[i, :R]):
                if float(sim) < min_sim:
                    continue
                jj = int(j)
                if 0 <= jj < len(self.knn_tgts):
                    t = str(self.knn_tgts[jj]).strip()
                    if t:
                        pools_txt[i].insert(0, t)
                        pools_tag[i].insert(0, "knn|raw")
                        added += 1
            if added:
                hit += 1
        return hit

    # -------------------------
    # public: eval hook (unchanged)
    # -------------------------
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        t0 = time.time()
        eval_dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
        if eval_dataset is None:
            raise ValueError("evaluate() needs an eval_dataset (even a placeholder) to produce eval_loss.")
        eval_dataloader = self.get_eval_dataloader(eval_dataset)
        output = self.evaluation_loop(
            eval_dataloader,
            description="Evaluation",
            prediction_loss_only=True,
            ignore_keys=ignore_keys,
            metric_key_prefix=metric_key_prefix,
        )
        metrics = dict(output.metrics)
        if self.is_world_process_zero():
            mbr_metrics = self._evaluate_mbr(metric_key_prefix=metric_key_prefix)
        else:
            mbr_metrics = None
        mbr_metrics = self._broadcast_metrics(mbr_metrics)
        if mbr_metrics is not None:
            metrics.update(mbr_metrics)
        metrics[f"{metric_key_prefix}_runtime"] = float(time.time() - t0)
        self.log(metrics)
        self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, metrics)
        return metrics

    # -------------------------
    # shared core: build pools + pick MBR
    # -------------------------
    @torch.inference_mode()
    def _mbr_from_sources(
        self,
        src_texts: list[str],
        *,
        ex_ids: list[str] | None = None,
        add_tbm: bool = True,
        use_knn: bool = True,
        return_pools: bool = False,
        return_tags: bool = False,
    ):
        model = self.model
        tok = getattr(self, "processing_class", None) or getattr(self, "tokenizer", None)
        if tok is None:
            raise ValueError("Tokenizer/processor not found (trainer.processing_class or trainer.tokenizer).")
        device = model.device

        src_clean = self.pre.preprocess_batch(list(map(str, src_texts)))

        knn_ids = knn_sims = knn_hints = None
        knn_on = bool(use_knn) and bool(self.has_knn)
        if knn_on:
            try:
                knn_ids, knn_sims, knn_hints = self._knn_compute(list(src_clean))
            except Exception as e:
                knn_ids = knn_sims = knn_hints = None
                knn_on = False
                if self.show_progress and self.is_world_process_zero():
                    print("[kNN] compute failed -> disabled for this call:", repr(e), flush=True)

        flat_inputs, flat_exi, flat_view = [], [], []
        for ex_i, base in enumerate(src_clean):
            base = str(base)
            seen_inp = set()

            def _add(txt: str, view: str):
                inp = self.prefix + str(txt)
                if inp in seen_inp:
                    return
                seen_inp.add(inp)
                flat_inputs.append(inp)
                flat_exi.append(ex_i)
                flat_view.append(view)

            _add(base, "raw")

            if self.has_pn:
                try:
                    pn = self.canon.canonicalize_source(base, mode=self.pn_mode)
                except Exception:
                    pn = base
                pn = "" if pn is None else str(pn)
                if pn and pn != base:
                    _add(pn, "pn")

            if self.has_gloss:
                if ex_ids is not None:
                    ex_int = _stable_int_id(str(ex_ids[ex_i]))
                else:
                    ex_int = _stable_int_id(f"infer::{ex_i}::{base[:48]}")
                for v in range(self.gloss_variants):
                    vseed = int(self.gloss_seed) + 1009 * int(v)
                    s_gl = self.glosser.append_gloss(
                        base,
                        max_items=self.gloss_max_items,
                        max_append_chars=self.gloss_max_append_chars,
                        seed=vseed,
                        epoch=0,
                        example_id=int(ex_int),
                        keep_order=True,
                    )
                    s_gl = "" if s_gl is None else str(s_gl)
                    if s_gl and s_gl != base:
                        _add(s_gl, "gloss")

            if knn_on and (knn_hints is not None):
                hint = str(knn_hints[ex_i] or "").strip()
                if hint:
                    s_knn = f"{base} <extra_id_1> TM: {hint}"
                    _add(s_knn, "knn0")

        lens = np.fromiter((len(x.split()) for x in flat_inputs), dtype=np.int32, count=len(flat_inputs))
        order = np.argsort(lens, kind="mergesort")

        pools_txt = [[] for _ in range(len(src_texts))]
        pools_tag = [[] for _ in range(len(src_texts))]

        pbar = self._tqdm(len(order), "MBR/generate")
        with self._bf16_eval_context():
            for a in range(0, len(order), self.mbr_batch_size_inputs):
                idx = order[a:a + self.mbr_batch_size_inputs]
                batch_in = [flat_inputs[i] for i in idx]
                batch_ex = [flat_exi[i] for i in idx]
                batch_vw = [flat_view[i] for i in idx]

                cand_txt_lists, cand_tag_lists = self._generate_multi_decode_tagged(
                    model, tok, batch_in, batch_vw,
                    device=device,
                )

                for ex_i2, vw, cands, tags in zip(batch_ex, batch_vw, cand_txt_lists, cand_tag_lists):
                    ex_i2 = int(ex_i2)
                    cands = list(cands or [])
                    tags = list(tags or [])
                    if len(tags) != len(cands):
                        tags = (tags + [f"other|{vw}"] * len(cands))[: len(cands)]
                    pools_txt[ex_i2].extend(cands)
                    pools_tag[ex_i2].extend(tags)

                if pbar is not None:
                    pbar.update(len(idx))
        if pbar is not None:
            pbar.close()

        pool_raw_mean = float(np.mean([len(p) for p in pools_txt])) if pools_txt else 0.0

        tbm_hit = 0
        if add_tbm and self.has_tbm:
            for ex_i, base_src in enumerate(src_clean):
                try:
                    res = self.tbm_index.query(str(base_src), k=self.tbm_topk)
                except Exception:
                    res = None
                if not res:
                    continue
                if float(res[0][1]) >= float(self.tbm_hard_sim):
                    prepend = [res[0][0]]
                else:
                    prepend = [t for (t, sim) in res if float(sim) >= float(self.tbm_min_sim)]
                if prepend:
                    pools_txt[ex_i] = prepend + pools_txt[ex_i]
                    pools_tag[ex_i] = (["tbm|raw"] * len(prepend)) + pools_tag[ex_i]
                    tbm_hit += 1

        knn_hit = 0
        if knn_on and (knn_ids is not None) and (knn_sims is not None):
            try:
                knn_hit = self._knn_inject_retrieval(pools_txt, pools_tag, knn_ids, knn_sims)
            except Exception:
                knn_hit = 0

        # NOTE: keep your existing tagged-dedupe helper if present; otherwise local fallback.
        dedup_tagged = globals().get("_dedup_keep_order_tagged", None)

        def _dedup_keep_order_tagged_local(xs, ts):
            if callable(dedup_tagged):
                return dedup_tagged(xs, ts)
            seen = set()
            xo, to = [], []
            for x, t in zip(xs, ts):
                if x in seen:
                    continue
                seen.add(x)
                xo.append(x)
                to.append(t)
            return xo, to

        pools_txt2, pools_tag2 = [], []
        for xs, ts in zip(pools_txt, pools_tag):
            xs2, ts2 = _dedup_keep_order_tagged_local(xs, ts)
            if self.mbr_pool_cap is not None:
                xs2 = xs2[: self.mbr_pool_cap]
                ts2 = ts2[: self.mbr_pool_cap]
            pools_txt2.append(xs2)
            pools_tag2.append(ts2)
        pools_txt, pools_tag = pools_txt2, pools_tag2

        flat_all, flat_tag, sizes = [], [], []
        for xs, ts in zip(pools_txt, pools_tag):
            sizes.append(len(xs))
            flat_all.extend(xs)
            flat_tag.extend(ts)

        if self.post is not None and flat_all:
            flat_all = self.post.postprocess_batch([str(x) for x in flat_all])
        flat_all = [_norm_ws(str(x)) for x in flat_all]

        pools_txt2, pools_tag2 = [], []
        k0 = 0
        for sz in sizes:
            xs = flat_all[k0:k0 + sz]
            ts = flat_tag[k0:k0 + sz]
            k0 += sz
            xs2, ts2 = _dedup_keep_order_tagged_local(xs, ts)
            pools_txt2.append(xs2)
            pools_tag2.append(ts2)
        pools_txt, pools_tag = pools_txt2, pools_tag2

        preds, chosen_tags, gaps = [], [], []
        pb_mbr = self._tqdm(len(pools_txt), "MBR/select")
        for xs, ts in zip(pools_txt, pools_tag):
            xs = list(xs)
            ts = list(ts)
            ts_pick = self._tags_for_pick(ts)
            n = len(xs)

            if n == 0:
                preds.append("")
                chosen_tags.append("")
                gaps.append(0.0)
            elif n == 1:
                preds.append(xs[0])
                chosen_tags.append(ts[0] if ts else "")
                gaps.append(0.0)
            else:
                best_i, gap_m, _ = self._pick_mbr(xs, ts_pick)
                bi = int(best_i)
                preds.append(xs[bi])
                chosen_tags.append(ts[bi] if bi < len(ts) else "")
                gaps.append(float(gap_m))

            if pb_mbr is not None:
                pb_mbr.update(1)
        if pb_mbr is not None:
            pb_mbr.close()

        preds = [_norm_ws(x) for x in preds]
        diag = {
            "pool_raw_mean": float(pool_raw_mean),
            "tbm_hit_rate": float(tbm_hit) / max(1, len(src_texts)),
            "knn_hit_rate": float(knn_hit) / max(1, len(src_texts)),
            "mbr_gap_mean": float(np.mean(gaps)) if len(gaps) else 0.0,
        }

        if return_pools and return_tags:
            return preds, pools_txt, pools_tag, chosen_tags, diag
        if return_pools:
            return preds, pools_txt, diag
        if return_tags:
            return preds, pools_tag, chosen_tags, diag
        return preds, diag

    # -------------------------
    # eval: uses val_text_ds and computes official metrics
    # -------------------------
    def _evaluate_mbr(self, metric_key_prefix="eval"):
        ex_ids, srcs, refs_raw = _val_unique_examples(self.val_text_ds, prefer_original=True)
        refs = [_norm_ws(r) for r in refs_raw]

        preds, pools_txt, pools_tag, chosen_tags, diag = self._mbr_from_sources(
            list(srcs),
            ex_ids=list(ex_ids),
            add_tbm=True,
            use_knn=True,
            return_pools=True,
            return_tags=True,
        )

        geo, bleu, chrfpp = _official_geo_mean(refs, preds)

        out = {
            f"{metric_key_prefix}_bleu": float(bleu),
            f"{metric_key_prefix}_chrfpp": float(chrfpp),
            f"{metric_key_prefix}_geo_mean": float(geo),

            # f"{metric_key_prefix}_pool_raw_mean": float(diag["pool_raw_mean"]),
            f"{metric_key_prefix}_tbm_hit_rate": float(diag.get("tbm_hit_rate", 0.0)),
            f"{metric_key_prefix}_knn_hit_rate": float(diag.get("knn_hit_rate", 0.0)),
            f"{metric_key_prefix}_mbr_gap_mean": float(diag["mbr_gap_mean"]),
        }

        # pick fractions + used-precision (adds knn view)
        denom = float(max(1, len(preds)))
        pick_origin = {"beam": 0, "samp": 0, "tbm": 0, "knn": 0, "other": 0}
        pick_view   = {"raw": 0, "pn": 0, "gloss": 0, "knn": 0, "other": 0}

        tbm_used_n = tbm_used_good = 0
        pn_used_n  = pn_used_good  = 0
        gloss_used_n = gloss_used_good = 0
        knn_used_n = knn_used_good = 0
        EPS = 1e-6

        for ref_i, pred_i, xs, ts, ctag in zip(refs, preds, pools_txt, pools_tag, chosen_tags):
            ref_i = _norm_ws(ref_i)
            pred_i = _norm_ws(pred_i)

            co = self._origin_from_tag_local(ctag)
            cv = self._view_from_tag_local(ctag)
            pick_origin[co if co in pick_origin else "other"] += 1
            pick_view[cv if cv in pick_view else "other"] += 1

            chosen_geo = float(_geo_sim_sentence(pred_i, ref_i)) if (pred_i and ref_i) else 0.0

            best_non_tbm = -1.0
            best_non_pn = -1.0
            best_non_gloss = -1.0
            best_non_knn = -1.0

            if ref_i:
                for c, tg in zip(xs, ts):
                    c = _norm_ws(str(c))
                    if not c:
                        continue
                    g = float(_geo_sim_sentence(c, ref_i))
                    o2 = self._origin_from_tag_local(tg)
                    v2 = self._view_from_tag_local(tg)

                    if o2 != "tbm":
                        best_non_tbm = max(best_non_tbm, g)
                    if v2 != "pn":
                        best_non_pn = max(best_non_pn, g)
                    if v2 != "gloss":
                        best_non_gloss = max(best_non_gloss, g)
                    if v2 != "knn":
                        best_non_knn = max(best_non_knn, g)

            if self.has_tbm and co == "tbm":
                tbm_used_n += 1
                if (best_non_tbm < 0) or (chosen_geo + EPS >= best_non_tbm):
                    tbm_used_good += 1

            if self.has_pn and cv == "pn":
                pn_used_n += 1
                if (best_non_pn < 0) or (chosen_geo + EPS >= best_non_pn):
                    pn_used_good += 1

            if self.has_gloss and cv == "gloss":
                gloss_used_n += 1
                if (best_non_gloss < 0) or (chosen_geo + EPS >= best_non_gloss):
                    gloss_used_good += 1

            if self.has_knn and cv == "knn":
                knn_used_n += 1
                if (best_non_knn < 0) or (chosen_geo + EPS >= best_non_knn):
                    knn_used_good += 1

        if self.has_tbm:
            out[f"{metric_key_prefix}_pick_tbm_frac"] = float(pick_origin["tbm"]) / denom
            out[f"{metric_key_prefix}_tbm_used_precision"] = float(tbm_used_good) / float(max(1, tbm_used_n))

        if self.has_pn:
            out[f"{metric_key_prefix}_pick_pn_frac"] = float(pick_view["pn"]) / denom
            out[f"{metric_key_prefix}_pn_used_precision"] = float(pn_used_good) / float(max(1, pn_used_n))

        if self.has_gloss:
            out[f"{metric_key_prefix}_pick_gloss_frac"] = float(pick_view["gloss"]) / denom
            out[f"{metric_key_prefix}_gloss_used_precision"] = float(gloss_used_good) / float(max(1, gloss_used_n))

        if self.has_knn:
            out[f"{metric_key_prefix}_pick_knn_frac"] = float(pick_view["knn"]) / denom
            out[f"{metric_key_prefix}_knn_used_precision"] = float(knn_used_good) / float(max(1, knn_used_n))

        return out

    # -------------------------
    # inference
    # -------------------------
    @torch.inference_mode()
    def mbr_predict(
        self,
        src_texts: list[str],
        *,
        add_tbm: bool = True,
        use_knn: bool = True,
    ):
        """
        Inference-time MBR (+ optional MSA polish).
        Returns ONLY preds (list[str]).
        """
        need_pool = bool(self.msa_enable)

        if not need_pool:
            preds, _diag = self._mbr_from_sources(
                list(src_texts),
                ex_ids=None,
                add_tbm=bool(add_tbm),
                use_knn=bool(use_knn),
                return_pools=False,
                return_tags=False,
            )
            return [_norm_ws(x) for x in preds]

        preds, pools, _diag = self._mbr_from_sources(
            list(src_texts),
            ex_ids=None,
            add_tbm=bool(add_tbm),
            use_knn=bool(use_knn),
            return_pools=True,
            return_tags=False,
        )

        _msa = globals().get("msa_consensus", None)
        if _msa is None:
            return [_norm_ws(x) for x in preds]

        def _mbr_gap_plain(p2: list[str]) -> float:
            n = len(p2)
            if n <= 1:
                return 0.0
            sums = np.zeros((n,), dtype=np.float32)
            for i in range(n):
                ai = p2[i]
                for j in range(i + 1, n):
                    s = _geo_sim_sentence(ai, p2[j])
                    sums[i] += s
                    sums[j] += s
            avg = sums / float(n - 1)
            jbest = int(np.argmax(avg))
            best = float(avg[jbest])
            tmp = avg.copy()
            tmp[jbest] = -1e9
            return float(best - float(np.max(tmp)))

        out = []
        for best, pool in zip(preds, pools):
            best = _norm_ws(str(best))
            pool = [_norm_ws(str(p)) for p in (pool or []) if str(p).strip()]

            if (not best) or (len(pool) < int(self.msa_min_pool)):
                out.append(best)
                continue

            if self.msa_gap_thr is not None:
                try:
                    gap = _mbr_gap_plain(_dedup_keep_order(pool))
                except Exception:
                    gap = 0.0
                if float(gap) > float(self.msa_gap_thr):
                    out.append(best)
                    continue

            try:
                out.append(_norm_ws(_msa(best, pool)))
            except Exception:
                out.append(best)

        return out

# MSA
# ============================================================
from collections import Counter
import math

def _maybe_tqdm(total: int, desc: str):
    try:
        from tqdm.auto import tqdm
        return tqdm(total=total, desc=desc)
    except Exception:
        return None

# --- Word-level MSA consensus (bioinformatics-inspired polishing) ---

def _char_bigram_sim(a: str, b: str) -> float:
    """Char-bigram Jaccard for fuzzy word matching in NW alignment."""
    a, b = a.lower(), b.lower()
    if a == b:
        return 1.0
    if not a or not b:
        return 0.0
    def _bg(s):
        return set(s[i:i+2] for i in range(len(s) - 1)) if len(s) > 1 else {s}
    ba, bb = _bg(a), _bg(b)
    inter = len(ba & bb)
    union = len(ba | bb)
    return inter / union if union > 0 else 0.0


def _nw_word_align(
    ref_toks: list[str],
    hyp_toks: list[str],
    *,
    match: float = 2.0,
    mismatch: float = -1.0,
    gap: float = -0.5,
    fuzzy_thr: float = 0.5,
) -> list[tuple]:
    """
    Needleman-Wunsch at word level with optional fuzzy partial credit.
    Returns list of (ref_pos|None, ref_tok|None, hyp_tok|None).
    """
    n, m = len(ref_toks), len(hyp_toks)

    # dp[i][j] = best score aligning ref[:i] to hyp[:j]
    dp = [[0.0] * (m + 1) for _ in range(n + 1)]
    bt = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0] = dp[i - 1][0] + gap
        bt[i][0] = 1  # up = deletion in hyp
    for j in range(1, m + 1):
        dp[0][j] = dp[0][j - 1] + gap
        bt[0][j] = 2  # left = insertion in hyp

    for i in range(1, n + 1):
        ri = ref_toks[i - 1]
        for j in range(1, m + 1):
            hj = hyp_toks[j - 1]
            if ri == hj:
                s = match
            elif _char_bigram_sim(ri, hj) >= fuzzy_thr:
                s = match * 0.5
            else:
                s = mismatch

            diag = dp[i - 1][j - 1] + s
            up = dp[i - 1][j] + gap
            left = dp[i][j - 1] + gap

            if diag >= up and diag >= left:
                dp[i][j] = diag; bt[i][j] = 0
            elif up >= left:
                dp[i][j] = up; bt[i][j] = 1
            else:
                dp[i][j] = left; bt[i][j] = 2

    # traceback
    aligned = []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and bt[i][j] == 0:
            aligned.append((i - 1, ref_toks[i - 1], hyp_toks[j - 1]))
            i -= 1; j -= 1
        elif i > 0 and (j == 0 or bt[i][j] == 1):
            aligned.append((i - 1, ref_toks[i - 1], None))
            i -= 1
        else:
            aligned.append((None, None, hyp_toks[j - 1]))
            j -= 1

    aligned.reverse()
    return aligned


def _build_ngram_counts(pool: list[str], max_n: int = 4) -> dict[tuple, int]:
    """
    Build n-gram frequency table from pool candidates.
    This is the "language model" derived from what the model actually generates.
    Analogous to k-mer spectrum in assembly.
    """
    counts: dict[tuple, int] = Counter()
    for cand in pool:
        toks = cand.split()
        for n in range(1, min(max_n, len(toks)) + 1):
            for i in range(len(toks) - n + 1):
                counts[tuple(toks[i:i+n])] += 1
    return counts


def _local_ngram_score(
    tokens: list[str],
    pos: int,
    ngram_counts: dict[tuple, int],
    max_n: int = 4,
) -> float:
    """
    Score how well position `pos` fits its context, using pool n-gram frequencies.
    Sums log(count+1) for all n-grams that include position `pos`.

    Higher = more candidates produced this exact n-gram context.
    """
    n_toks = len(tokens)
    score = 0.0
    for n in range(1, min(max_n, n_toks) + 1):
        # all n-grams that include position `pos`
        start_lo = max(0, pos - n + 1)
        start_hi = min(pos, n_toks - n)
        for start in range(start_lo, start_hi + 1):
            ng = tuple(tokens[start:start + n])
            cnt = ngram_counts.get(ng, 0)
            # weight higher n-grams more (they're more informative)
            w = float(n)
            score += w * math.log1p(cnt)
    return score


def msa_consensus(
    ref_text: str,
    pool: list[str],
    *,
    min_agreement: float = 0.35,
    min_pool: int = 3,
    tie_bias: int = 2,
    fuzzy_thr: float = 0.5,
    max_insert_len: int = 3,
    # --- n-gram aware params ---
    ngram_verify: bool = True,
    ngram_max_n: int = 4,
    ngram_min_gain: float = 0.0,  # accept change only if ngram score improves by at least this
    mbr_verify: bool = True,      # final check: only accept if pool-agreement doesn't drop
) -> str:
    """
    Word-level MSA consensus with n-gram-aware edit acceptance.

    Phase 1: Standard NW alignment + per-position voting (same as before)
    Phase 2: N-gram verification — for each proposed word change, check whether
             the surrounding n-grams have better support in the pool.
             Rejects edits that break high-frequency n-gram contexts.
    Phase 3: MBR verification — score final output against pool, reject if worse.

    The n-gram table acts as a "k-mer spectrum" — it captures which word sequences
    the model actually co-generates, not just individual word frequencies.
    """
    pool = [str(c).strip() for c in pool if str(c).strip()]
    ref_text = str(ref_text).strip()
    ref_toks = ref_text.split()

    if not ref_toks or len(pool) < int(min_pool):
        return ref_text

    n_ref = len(ref_toks)
    n_cands = len(pool)
    threshold = max(2, int(n_cands * float(min_agreement)))

    # ================================================================
    # Phase 1: Standard alignment + voting (unchanged logic)
    # ================================================================
    position_votes = [Counter() for _ in range(n_ref)]
    insert_counter = Counter()

    for cand in pool:
        cand_toks = cand.split()
        if not cand_toks:
            continue

        aligned = _nw_word_align(ref_toks, cand_toks, fuzzy_thr=float(fuzzy_thr))
        insert_buf = []

        for ref_pos, ref_tok, hyp_tok in aligned:
            if ref_pos is not None:
                if insert_buf and len(insert_buf) <= int(max_insert_len):
                    insert_counter[(ref_pos, tuple(insert_buf))] += 1
                insert_buf = []
                position_votes[ref_pos][hyp_tok] += 1
            else:
                insert_buf.append(hyp_tok)

        if insert_buf and len(insert_buf) <= int(max_insert_len):
            insert_counter[(n_ref, tuple(insert_buf))] += 1

    # ================================================================
    # Build proposed edits (what standard MSA would produce)
    # ================================================================
    # For each position, determine: keep ref, substitute, or delete
    proposed = []  # list of (action, token_or_None)
    #   action: "keep", "sub", "del"

    for i in range(n_ref):
        votes = position_votes[i]
        if not votes:
            proposed.append(("keep", ref_toks[i]))
            continue

        best_tok, best_count = votes.most_common(1)[0]

        if best_tok is None:
            # deletion candidate
            if best_count >= threshold:
                proposed.append(("del", None))
            else:
                proposed.append(("keep", ref_toks[i]))
        else:
            ref_count = votes.get(ref_toks[i], 0)
            if best_tok != ref_toks[i] and (best_count - ref_count) >= int(tie_bias):
                proposed.append(("sub", best_tok))
            else:
                proposed.append(("keep", ref_toks[i]))

    # Collect proposed insertions (same as before)
    proposed_inserts = {}  # pos -> list of tokens
    for i in range(n_ref + 1):
        best_ins = None
        best_ins_count = 0
        for (pos, toks), cnt in insert_counter.items():
            if pos == i and cnt > best_ins_count:
                best_ins = toks
                best_ins_count = cnt
        if best_ins is not None and best_ins_count >= threshold:
            proposed_inserts[i] = list(best_ins)

    # ================================================================
    # Phase 2: N-gram verification of each edit
    # ================================================================
    if ngram_verify and pool:
        ngram_counts = _build_ngram_counts(pool, max_n=int(ngram_max_n))

        # Check each substitution/deletion against n-gram context
        for i in range(n_ref):
            action, tok = proposed[i]

            if action == "keep":
                continue  # no change to verify

            # Build ref version (with ref token at position i)
            ref_version = []
            for j in range(n_ref):
                if j == i:
                    ref_version.append(ref_toks[j])
                else:
                    _, t = proposed[j]
                    if t is not None:
                        ref_version.append(t)
                    else:
                        ref_version.append(ref_toks[j])  # placeholder for scoring

            # Build proposed version
            prop_version = list(ref_version)
            if action == "sub":
                prop_version[i] = tok
            elif action == "del":
                # for scoring, treat deletion as removing the token
                # score both: with token present vs absent
                pass

            # Find position in the built sequence (accounting for earlier deletions)
            # Simpler: just score with token present in both versions at position i
            if action == "sub":
                ref_score = _local_ngram_score(ref_version, i, ngram_counts, max_n=int(ngram_max_n))
                prop_score = _local_ngram_score(prop_version, i, ngram_counts, max_n=int(ngram_max_n))

                if prop_score < ref_score + float(ngram_min_gain):
                    # n-gram context is worse -> revert to ref token
                    proposed[i] = ("keep", ref_toks[i])

            elif action == "del":
                # For deletion: compare n-grams with vs without the token
                with_tok = list(ref_version)  # has ref token at i
                without_tok = [t for j, t in enumerate(ref_version) if j != i]

                # Score the bigram/trigram spanning the deletion point
                if i > 0 and i < len(without_tok):
                    # n-grams that would bridge the gap
                    bridge_score = 0.0
                    for n in range(2, min(int(ngram_max_n), len(without_tok)) + 1):
                        start = max(0, i - n + 1)
                        end = min(i + 1, len(without_tok) - n + 1)
                        for s in range(start, end):
                            ng = tuple(without_tok[s:s+n])
                            bridge_score += float(n) * math.log1p(ngram_counts.get(ng, 0))

                    # n-grams with the token present
                    keep_score = _local_ngram_score(with_tok, i, ngram_counts, max_n=int(ngram_max_n))

                    if bridge_score < keep_score + float(ngram_min_gain):
                        proposed[i] = ("keep", ref_toks[i])

        # Verify insertions: check if inserted n-grams appear in pool
        inserts_to_remove = []
        for pos, ins_toks in proposed_inserts.items():
            # Build local context around insertion point
            before = []
            for j in range(max(0, pos - 2), pos):
                _, t = proposed[j]
                if t is not None:
                    before.append(t)

            after = []
            for j in range(pos, min(n_ref, pos + 2)):
                _, t = proposed[j]
                if t is not None:
                    after.append(t)

            # Check if the inserted phrase + context appears in pool n-grams
            test_seq = before + ins_toks + after
            if len(test_seq) >= 2:
                ins_support = 0.0
                for n in range(2, min(int(ngram_max_n), len(test_seq)) + 1):
                    for s in range(len(test_seq) - n + 1):
                        ng = tuple(test_seq[s:s+n])
                        ins_support += float(n) * math.log1p(ngram_counts.get(ng, 0))

                # Compare to context without insertion
                no_ins_seq = before + after
                no_ins_support = 0.0
                if len(no_ins_seq) >= 2:
                    for n in range(2, min(int(ngram_max_n), len(no_ins_seq)) + 1):
                        for s in range(len(no_ins_seq) - n + 1):
                            ng = tuple(no_ins_seq[s:s+n])
                            no_ins_support += float(n) * math.log1p(ngram_counts.get(ng, 0))

                if ins_support < no_ins_support + float(ngram_min_gain):
                    inserts_to_remove.append(pos)

        for pos in inserts_to_remove:
            del proposed_inserts[pos]

    # ================================================================
    # Emit final consensus
    # ================================================================
    result = []
    for i in range(n_ref):
        # insertions before position i
        if i in proposed_inserts:
            result.extend(proposed_inserts[i])

        action, tok = proposed[i]
        if action == "del":
            continue
        result.append(tok if tok is not None else ref_toks[i])

    # trailing insertions
    if n_ref in proposed_inserts:
        result.extend(proposed_inserts[n_ref])

    consensus = " ".join(result)

    # ================================================================
    # Phase 3: MBR verification (optional safety net)
    # ================================================================
    if mbr_verify and consensus != ref_text and pool:
        # Score both against pool using same geo_sim as MBR
        ref_pool_score = sum(_geo_sim_sentence(ref_text, c) for c in pool) / len(pool)
        con_pool_score = sum(_geo_sim_sentence(consensus, c) for c in pool) / len(pool)

        if con_pool_score < ref_pool_score:
            return ref_text

    # Safety fallback
    if not consensus.strip() or len(consensus.split()) < len(ref_toks) * 0.3:
        return ref_text

    return consensus


# GlossAugmenter and PN canon
# ------------------------------------------------------------
# punctuation strip used by multiple helpers 
_PUNCT_STRIP = "[](){}<>.,;:!?\"'“”„`´"

_SUBSCRIPT_TRANS = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
_H_HAT_TRANS     = str.maketrans("ḫḪ", "hH")

_ROMAN_TAIL_RE   = re.compile(r"\s+[IVX]+$")                      # trailing I/II/III...
_ROMAN_ANY_RE    = re.compile(r"\s+(I|II|III|IV|V|VI|VII|VIII|IX|X)\b.*$")

_QUOTED_RE       = re.compile(r'"([^"]{1,80})"')
_DET_RE          = re.compile(r"(^|\-)\((d|m|f)\)", re.I)
_ELLIPSIS_RE     = re.compile(r"(…|\.\.\.)")
_NUM_RE          = re.compile(r"^\d+(\.\d+)?$")

_SUB_MAP         = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")

# conservative suffixes used by NB-ish candidate generation
_AKK_SUFFIXES = [
    "šu-nu", "šunu", "šunū",
    "šu", "ša", "ši", "šū",
    "ī",
]

def _key(s: str) -> str:
    # Keep case, normalize subscripts + ḫ.
    s = "" if s is None else str(s)
    return s.translate(_SUBSCRIPT_TRANS).translate(_H_HAT_TRANS).strip()

def _split_spellings(cell: str) -> List[str]:
    if not isinstance(cell, str) or not cell.strip():
        return []
    parts = re.split(r"[;,\|/]+", cell)
    return [p.strip() for p in parts if p.strip()]

def _lemma_part(x: str) -> Optional[str]:
    # stable lemma key: remove trailing roman numerals; keep leftmost chunk
    if x is None:
        return None
    s = str(x).strip()
    if not s or s.lower() == "nan":
        return None
    s = _ROMAN_TAIL_RE.sub("", s).strip()
    s = re.split(r"[\s/;]", s, maxsplit=1)[0].strip()
    return s if s else None

def _first_quoted_gloss(defn: str) -> Optional[str]:
    if defn is None:
        return None
    s = str(defn).strip()
    if not s or s.lower() == "nan":
        return None
    m = _QUOTED_RE.search(s)
    if not m:
        return None
    g = m.group(1).strip()
    if not g:
        return None
    # keep short; avoid long prose
    g = g.replace(",", " ").replace("/", " ")
    g = " ".join(g.split()[:4])
    return g or None

def _stable_u32(s: str) -> int:
    b = ("" if s is None else str(s)).encode("utf-8", errors="ignore")
    return zlib.crc32(b) & 0xFFFFFFFF

def _is_num(tok: str) -> bool:
    return bool(_NUM_RE.match("" if tok is None else str(tok).strip()))

def _norm(tok: str) -> str:
    tok = "" if tok is None else str(tok)
    tok = tok.strip().lower()
    tok = unicodedata.normalize("NFKC", tok)
    tok = tok.strip(".,;:()[]{}")
    if tok == "...":
        tok = "…"
    return tok

# ------------------------------------------------------------
# 3) Simple eBL dictionary -> glossary injection (same public names)
# ------------------------------------------------------------
EBL_PATH = Config.EBL_DICT_PATH

def _clean_word(w: str) -> str:
    w = "" if w is None else str(w).strip()
    return _ROMAN_TAIL_RE.sub("", w).strip()

def _short_gloss(defn: str) -> str | None:
    if defn is None or (isinstance(defn, float) and pd.isna(defn)):
        return None
    s = str(defn).strip()
    if not s:
        return None
    m = _QUOTED_RE.search(s)
    if m:
        g = m.group(1).strip()
        if g:
            return g
    s = re.split(r"[;(]", s, maxsplit=1)[0].strip()
    s = re.sub(r"\s+", " ", s).lstrip("= ").strip()
    words = s.split()
    return " ".join(words[:6]) if words else None

def load_ebl_lexicon(path=EBL_PATH, min_len=2):
    df = pd.read_csv(path)
    lex = defaultdict(list)

    for w, d, der in zip(df.get("word", []), df.get("definition", []), df.get("derived_from", [])):
        w = _clean_word(w)
        if len(w) < int(min_len):
            continue
        g = _short_gloss(d)
        if g:
            lex[w].append(g)

        if der is not None and not (isinstance(der, float) and pd.isna(der)):
            der = _clean_word(der)
            if der and g:
                lex[der].append(g)

    return {k: list(dict.fromkeys(v)) for k, v in lex.items()}

def normalize_src_token(tok: str) -> str:
    t = ("" if tok is None else str(tok)).strip().strip(_PUNCT_STRIP)
    return re.sub(r"\d+$", "", t)

def find_glossary_terms(src: str, lex: dict, max_terms=8):
    toks = [normalize_src_token(t) for t in ("" if src is None else str(src)).split()]
    hits, seen = [], set()
    for t in toks:
        if not t or t in seen:
            continue
        if t in lex:
            hits.append(t)
            seen.add(t)
            if len(hits) >= int(max_terms):
                break
    return hits

def add_glossary_to_source(src: str, lex: dict, max_terms=8, drop_prob=0.5):
    # IMPORTANT: never include literal </s> for ByT5 (EOS). Use <extra_id_0>.
    src = "" if src is None else str(src)
    if drop_prob and random.random() < float(drop_prob):
        return src

    terms = find_glossary_terms(src, lex, max_terms=max_terms)
    if not terms:
        return src

    items = []
    for t in terms:
        if t in lex and lex[t]:
            items.append(f"{t}={lex[t][0]}")
    if not items:
        return src

    return f"{src} <extra_id_0> GLOSSARY: " + " ; ".join(items)

# ------------------------------------------------------------
# 4) NB-ish surface normalization + candidate generation (kept)
# ------------------------------------------------------------
def _norm_form(s: str) -> str:
    s = "" if s is None else str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("–", "-").replace("—", "-")
    s = s.strip().strip(_PUNCT_STRIP)
    s = _DET_RE.sub(r"\1", s)
    s = s.replace("[", "").replace("]", "")
    s = s.translate(_SUB_MAP)
    s = _ELLIPSIS_RE.sub("", s)
    return s.strip()

def _suffix_strips(s: str, *, max_depth: int = 2) -> list[str]:
    out = set()
    cur = {_norm_form(s)}
    for _ in range(int(max_depth)):
        nxt = set()
        for x in cur:
            for suf in _AKK_SUFFIXES:
                for lead in ("-", ""):
                    tail = lead + suf
                    if x.endswith(tail) and len(x) > len(tail) + 2:
                        base = x[:-len(tail)].rstrip("-")
                        if base and base not in out:
                            nxt.add(base)
        out |= nxt
        cur = nxt
        if not cur:
            break
    return list(out)

def _candidates_for_form(t: str) -> list[str]:
    raw = "" if t is None else str(t).strip()
    if not raw:
        return []

    cands, seen = [], set()
    def add(x):
        x = "" if x is None else str(x).strip()
        if x and x not in seen:
            seen.add(x)
            cands.append(x)

    add(raw)
    n0 = _norm_form(raw)
    add(n0)
    add(raw.translate(_SUB_MAP))
    for v in _suffix_strips(n0, max_depth=2):
        add(v)
        add(v.translate(_SUB_MAP))
    return cands

def _is_junk_surface(surface: str) -> bool:
    s = "" if surface is None else str(surface).strip()
    if not s:
        return True
    if s in {"<gap>", "<big_gap>", "x", "x.", "…", "...", "[...]", "…]", "[…"}:
        return True
    if re.fullmatch(r"\d+([.:]\d+)?", s):
        return True
    core = s.strip(_PUNCT_STRIP)
    return not bool(core)



class GlossAugmenter:
    """
    Uses OA_Lexicon to map source 'form' -> 'lexeme',
    then uses eBL_Dictionary to map lemma(lexeme) -> short English gloss.

    Updates:
    - glossary key uses the ORIGINAL src token that matched (not the normalized candidate)
    - gloss is shortened to 1 head-gloss (no long synonym strings)
    - unseen df==0 is NOT clamped by rare_df_floor (enhance unseen, still capped by idf_cap)
    - safe delimiter: "<extra_id_0>"
    """
    def __init__(
        self,
        oa_lexicon_path: str,
        ebl_dict_path: str,
        *,
        train_texts: Optional[list[str]] = None,
        idf_cap: float = 3.5,
        rare_df_floor: int = 3,
        df1_penalty: float = 0.65,
        base_weight: float = 0.01,

        # new (safe defaults)
        gloss_max_chars: int = 48,     # cap per-gloss text
        unseen_boost: float = 1.15,    # >1 boosts df==0 slightly
    ):
        lex = pd.read_csv(oa_lexicon_path)
        dic = pd.read_csv(ebl_dict_path)

        def _short_gloss(g: str) -> str:
            g = "" if g is None else str(g).strip()
            if not g:
                return ""
            # drop leading parenthetical: "(mythical ...) snake" -> "snake"
            g = re.sub(r"^\([^)]*\)\s*", "", g).strip()

            # keep only first sense
            # - split on ';' first (often separates senses)
            # - then split on ',' (often adds extra detail)
            g = g.split(";", 1)[0].strip()
            g = g.split(",", 1)[0].strip()

            # drop remaining parentheticals (optional but helps)
            g = re.sub(r"\([^)]*\)", "", g).strip()

            g = _norm_ws(g)
            if int(gloss_max_chars) > 0 and len(g) > int(gloss_max_chars):
                g = g[: int(gloss_max_chars)].rstrip()
            return g

        lemma2gloss: Dict[str, str] = {}
        for w, d in zip(dic["word"].astype(str), dic["definition"].astype(str)):
            lemma = _lemma_part(w)
            g0 = _first_quoted_gloss(d)
            gloss = _short_gloss(g0)
            if lemma and gloss and lemma not in lemma2gloss:
                lemma2gloss[lemma] = gloss

        self.form2lex = dict(zip(lex["form"].astype(str), lex["lexeme"].astype(str)))
        self.lex2gloss = lemma2gloss

        self._df: Dict[str, int] = {}
        self._N: int = 0

        if train_texts:
            from collections import Counter
            dfc = Counter()
            N = 0
            for s in train_texts:
                toks = ("" if s is None else str(s)).split()
                seen = set()
                for t in toks:
                    for cand in _candidates_for_form(t):
                        if cand:
                            seen.add(cand)
                for x in seen:
                    dfc[x] += 1
                N += 1
            self._df = dict(dfc)
            self._N = int(N)
        else:
            vc = lex["form"].astype(str).value_counts()
            self._df = {k: int(v) for k, v in vc.items()}
            self._N = int(vc.sum())

        self._idf_cap = float(idf_cap)
        self._rare_df_floor = int(rare_df_floor)
        self._df1_penalty = float(df1_penalty)
        self._base_weight = float(base_weight)
        self._unseen_boost = float(unseen_boost)

    def _weight_for_surface(self, surface: str) -> float:
        if _is_junk_surface(surface):
            return 0.0
        s = _norm_form(surface)
        df = int(self._df.get(s, 0))

        # IMPORTANT:
        # - unseen df==0: do NOT clamp to rare_df_floor (enhance unseen)
        # - seen df>0: you may clamp low dfs to avoid rare garbage
        if df <= 0:
            df_eff = 1
        else:
            df_eff = max(df, int(self._rare_df_floor))

        idf = math.log((self._N + 1.0) / (df_eff + 1.0))
        idf = max(0.0, min(float(idf), float(self._idf_cap)))

        w = float(self._base_weight) + idf

        # penalize df==1 (rare-but-seen) only; don't penalize unseen
        if df == 1:
            w *= float(self._df1_penalty)

        # boost unseen a bit (still capped by idf_cap)
        if df <= 0:
            w *= float(self._unseen_boost)

        # numbers tend to be noisy
        if re.search(r"\d{2,}", str(surface)):
            w *= 0.6

        return max(0.0, float(w))

    def append_gloss(
        self,
        src_text: str,
        max_items: int = 6,
        max_append_chars: int = 240,
        *,
        seed: int = 0,
        epoch: int = 0,
        example_id: Optional[int] = None,
        keep_order: bool = True,
    ) -> str:
        src_text = "" if src_text is None else str(src_text)

        # never include literal </s>
        if "</s>" in src_text:
            src_text = src_text.replace("</s>", "<eos>")

        if (not src_text.strip()) or int(max_items) <= 0 or int(max_append_chars) <= 0:
            return src_text

        toks = src_text.split()
        candidates = []  # (pos, src_tok, match_surface, gloss, weight)
        seen = set()

        for pos, t in enumerate(toks):
            if _is_junk_surface(t):
                continue

            lexeme = None
            match_surface = None
            for cand in _candidates_for_form(t):
                lexeme = self.form2lex.get(cand)
                if lexeme:
                    match_surface = cand
                    break
            if not lexeme:
                continue

            lemma = _lemma_part(lexeme)
            if not lemma:
                continue

            g = self.lex2gloss.get(lemma)
            if not g:
                continue

            # dedupe by (normalized src token, lemma, gloss)
            key = (_norm_form(t), lemma, g)
            if key in seen:
                continue
            seen.add(key)

            w = self._weight_for_surface(match_surface)
            if w <= 0:
                continue

            candidates.append((pos, t, match_surface, g, w))

        if not candidates:
            return src_text

        ex = int(example_id) if example_id is not None else 0
        mix = (_stable_u32(src_text) ^ int(seed) ^ (int(epoch) * 1000003) ^ (ex * 9176)) & 0xFFFFFFFF
        rng = random.Random(mix)

        k = min(int(max_items), len(candidates))

        # weighted sampling without replacement (Efraimidis–Spirakis)
        keys = []
        for j, (_, _, _, _, w) in enumerate(candidates):
            u = max(1e-12, rng.random())
            keys.append((-math.log(u) / max(1e-6, float(w)), j))
        keys.sort(key=lambda x: x[0])
        picked = [candidates[j] for _, j in keys[:k]]

        if keep_order:
            picked.sort(key=lambda x: x[0])

        parts, used = [], 0
        for _, src_tok, _match_surface, gloss, _w in picked:
            # KEY FIX: emit the ORIGINAL src token that matched
            part = f"{src_tok}={gloss}"
            add_len = len(part) + (3 if parts else 0)
            if used + add_len > int(max_append_chars):
                break
            parts.append(part)
            used += add_len

        if not parts:
            return src_text

        return src_text + " <extra_id_0> GLOSSARY: " + " ; ".join(parts)
    
# ------------------------------------------------------------
# 6) SourceCanonicalizer (same class + methods)
# ------------------------------------------------------------
@dataclass
class SourceCanonicalizer:
    pn_gn_map: Dict[str, str]
    ono_map: Dict[str, str]

    @classmethod
    def from_csvs(
        cls,
        lexicon_path: str,
        onomasticon_path: str,
        use_norm: bool = True,
    ) -> "SourceCanonicalizer":
        lex = pd.read_csv(lexicon_path)
        ono = pd.read_csv(onomasticon_path)

        lex = lex[lex["type"].isin(["PN", "GN"])].copy()
        lex["form"] = lex["form"].astype(str)
        lex["canon"] = (lex["norm"].astype(str) if use_norm else lex["lexeme"].astype(str))

        lex["canon_len"] = lex["canon"].str.len()
        lex = lex.sort_values(["form", "canon_len"]).drop_duplicates("form", keep="first")

        pn_gn_map: Dict[str, str] = {}
        for form, canon in zip(lex["form"].tolist(), lex["canon"].tolist()):
            pn_gn_map[_key(form)] = canon

        if "Alt_lex" in lex.columns:
            alt_src = pd.read_csv(lexicon_path)
            alt_src = alt_src[alt_src["type"].isin(["PN", "GN"])].copy()
            alt_src["form"] = alt_src["form"].astype(str)
            alt_src["canon"] = (alt_src["norm"].astype(str) if use_norm else alt_src["lexeme"].astype(str))
            for form, canon, alt in zip(
                alt_src["form"],
                alt_src["canon"],
                alt_src.get("Alt_lex", pd.Series([None] * len(alt_src))),
            ):
                for v in _split_spellings(alt):
                    pn_gn_map.setdefault(_key(v), canon)

        ono_map: Dict[str, str] = {}
        if {"Name", "Spellings_semicolon_separated"}.issubset(set(ono.columns)):
            for name, cell in zip(ono["Name"].astype(str), ono["Spellings_semicolon_separated"]):
                for v in _split_spellings(cell):
                    ono_map[_key(v)] = name
            for name in ono["Name"].astype(str).tolist():
                ono_map.setdefault(_key(name), name)

        return cls(pn_gn_map=pn_gn_map, ono_map=ono_map)

    def canonicalize_source(self, text: str, mode: str = "pn_norm") -> str:
        if mode == "original" or not isinstance(text, str) or not text.strip():
            return text if isinstance(text, str) else ""
        out = []
        for t in text.split():
            kt = _key(t)
            out.append(self.pn_gn_map.get(kt) or self.ono_map.get(kt) or t)
        return " ".join(out)

    def extract_canonical_names_from_source(self, text: str) -> Set[str]:
        names: Set[str] = set()
        if not isinstance(text, str) or not text.strip():
            return names
        for t in text.split():
            kt = _key(t)
            if kt in self.pn_gn_map:
                names.add(self.pn_gn_map[kt])
            elif kt in self.ono_map:
                names.add(self.ono_map[kt])
        return names


# =========================
# 2) Load model + helpers (glosser joblib + TBM pairs.csv) + run MBRGlossSeq2SeqTrainer.mbr_predict
# =========================

# Load model/tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(
    cfg.MODEL_DIR,
    dtype=dtype,
    low_cpu_mem_usage=True,
)
model.to(device)
model.eval()
print("model dtype:", next(model.parameters()).dtype)

# Pre/Post (infer mode)
pre = OptimizedPreprocessor(mode="infer")
post_out = VectorizedPostprocessor(mode="infer", aggressive=True, enable_host_rewrites=False)

# Glosser: load (already built)
glosser = None
if bool(getattr(cfg, "GLOSS_ENABLE", False)):
    jl = str(getattr(cfg, "GLOSSER_JL_PATH", "") or "")
    if jl and os.path.exists(jl):
        glosser = joblib.load(jl)
        print(f"[GLOSS] enabled <- {jl}", flush=True)
    else:
        print("[GLOSS] disabled (missing cfg.GLOSSER_JL_PATH)", flush=True)
else:
    print("[GLOSS] disabled (GLOSS_ENABLE=False)", flush=True)

# TBM pairs.csv (train-split pairs exported by training NB)
tbm_pairs_df = None
if bool(getattr(cfg, "TBM_ENABLE", True)):
    pth = str(getattr(cfg, "TBM_PAIRS_PATH", "") or "")
    if pth and os.path.exists(pth):
        tbm_pairs_df = pd.read_csv(pth)
        print(f"[TBM] pairs loaded: {len(tbm_pairs_df)} <- {pth}", flush=True)
    else:
        print("[TBM] disabled (missing TBM_PAIRS_PATH)", flush=True)

# Minimal Trainer args (we only use mbr_predict)
use_bf16 = bool(torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)())
args = Seq2SeqTrainingArguments(
    output_dir=str(getattr(cfg, "OUTPUT_DIR", "/kaggle/working")),
    report_to="none",
    fp16=False,
    bf16=use_bf16,
    per_device_eval_batch_size=1,
    dataloader_num_workers=0,
)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# dummy val_text_ds (required by trainer __init__, not used in inference)
train_csv = os.path.join(cfg.INPUT_DIR, "train.csv")
if os.path.exists(train_csv):
    _val_df = pd.read_csv(train_csv, nrows=8)[["transliteration","translation"]].copy()
else:
    _val_df = pd.DataFrame({"transliteration":[""], "translation":[""]})

trainer = MBRGlossSeq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=None,
    eval_dataset=None,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=None,

    val_text_ds=_val_df,
    pre=pre,
    prefix=cfg.PREFIX,
    post=post_out,
    post_ref=None,

    glosser=glosser,
    gloss_variants=int(getattr(cfg, "MBR_GLOSS_VARIANTS", 1)) if (glosser is not None) else 0,
    gloss_seed=int(getattr(cfg, "SEED", cfg.SEED)) + 777,
    gloss_max_items=int(getattr(cfg, "GLOSS_MAX_ITEMS", 6)),
    gloss_max_append_chars=int(getattr(cfg, "GLOSS_MAX_APPEND_CHARS", 240)),

    canon=None,
    pn_enable=False,

    mbr_batch_size_inputs=int(getattr(cfg, "BATCH_INPUTS", 16)),
    src_max_length=int(getattr(cfg, "SRC_MAX_LENGTH", 512)),
    max_new_tokens=int(getattr(cfg, "GEN_MAX_NEW_TOKENS", 512)),

    num_beams=int(getattr(cfg, "NUM_BEAMS", 4)),
    num_beam_cands=int(getattr(cfg, "NUM_BEAM_CANDS", 1)),
    num_sample_cands=int(getattr(cfg, "NUM_SAMPLE_CANDS", 6)),
    mbr_pool_cap=int(getattr(cfg, "MBR_POOL_CAP", 36)),
    length_penalty=float(getattr(cfg, "LENGTH_PENALTY", 1.2)),
    temperature=float(getattr(cfg, "TEMPERATURE", 0.75)),
    top_p=float(getattr(cfg, "TOP_P", 0.9)),
    repetition_penalty=float(getattr(cfg, "REPETITION_PENALTY", 1.1)),
    no_repeat_ngram_size=int(getattr(cfg, "NO_REPEAT_NGRAM", 0)) or 0,

    tbm_pairs=tbm_pairs_df,
    tbm_enable=bool(getattr(cfg, "TBM_ENABLE", True)) and (tbm_pairs_df is not None),
    tbm_topk=int(getattr(cfg, "TBM_TOPK", 3)),
    tbm_min_sim=float(getattr(cfg, "TBM_MIN_SIM", 0.90)),
    tbm_hard_sim=float(getattr(cfg, "TBM_HARD_SIM", 0.995)),
    tbm_ngram=(int(getattr(cfg, "TBM_NGRAM_MIN", 3)), int(getattr(cfg, "TBM_NGRAM_MAX", 6))),
    tbm_max_features=int(getattr(cfg, "TBM_MAX_FEATURES", 250_000)),

    msa_enable=bool(getattr(cfg, "USE_MSA_POLISH", False)),
    msa_gap_thr=getattr(cfg, "MSA_GAP_THR", None),
    msa_min_pool=int(getattr(cfg, "MSA_MIN_POOL", 3)),
)

# =========================
# 3) Inference + submission
# =========================
test_df = pd.read_csv(cfg.TEST_PATH)
sub_df  = pd.read_csv(cfg.SUB_PATH)

ids = test_df["id"].astype(str).tolist()
srcs = test_df["transliteration"].astype(str).tolist()

preds_all = []
BEX = int(getattr(cfg, "BATCH_EXAMPLES", 16))
pbar = tqdm(total=len(srcs), desc="Infer", leave=True)

for a in range(0, len(srcs), BEX):
    batch_src = srcs[a:a+BEX]
    preds = trainer.mbr_predict(
        batch_src,
        add_tbm=bool(getattr(cfg, "TBM_ENABLE", True)) and (tbm_pairs_df is not None),
        use_knn=False,
    )
    preds_all.extend(preds)
    pbar.update(len(batch_src))

pbar.close()

sub_df["translation"] = preds_all
out_path = "./submission.csv"
sub_df[["id","translation"]].to_csv(out_path, index=False)
print("Wrote:", out_path)
sub_df.head()

2026-03-05 04:48:12.831892: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772686093.035039      31 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772686093.095713      31 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772686093.576601      31 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772686093.576638      31 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772686093.576641      31 computation_placer.cc:177] computation placer alr

torch: 2.8.0+cu126
transformers: 4.57.1
sacrebleu: 2.6.0
device: cuda dtype: torch.bfloat16
model dtype: torch.bfloat16
[GLOSS] disabled (missing cfg.GLOSSER_JL_PATH)
[TBM] disabled (missing TBM_PAIRS_PATH)


Infer:   0%|          | 0/4 [00:00<?, ?it/s]

MBR/generate:   0%|          | 0/4 [00:00<?, ?it/s]

MBR/select:   0%|          | 0/4 [00:00<?, ?it/s]

Wrote: ./submission.csv


,id,translation
0,0,"Thus kārum Kanesh, say to the <gap> -dātum of ..."
1,1,In the tablet you wrote to the City. On this d...
2,2,"As you hear our letter, whether he gives anyth..."
3,3,I sent it to every single and to the trading s...
